In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset
df_final = pd.read_csv("final_processed_tabular.csv")
# Define features (X) and target (y)
# We drop 'lat' and 'long' for the pure tabular baseline as they are geographical
X = df_final.drop(columns=['log_price', 'lat', 'long'])
y = df_final['log_price']

# Split: 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale data (Essential for KNN and helpful for some boosting models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 2. XGBoost
xgb = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42)
xgb.fit(X_train, y_train)

# 3. CatBoost (Handles categorical cues well internally)
cat = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, verbose=0)
cat.fit(X_train, y_train)

In [4]:
from sklearn.neighbors import KNeighborsRegressor
import numpy as np

# Generate "Level 1" predictions (Validation set predictions)
rf_preds = rf.predict(X_test)
xgb_preds = xgb.predict(X_test)
cat_preds = cat.predict(X_test)

# Stack these predictions to create a new feature set for the Meta-Model
stacked_predictions = np.column_stack((rf_preds, xgb_preds, cat_preds))

# Initialize and train KNN as the Meta-Learner
# KNN will learn which model is most reliable for different types of properties
meta_model = KNeighborsRegressor(n_neighbors=5)
meta_model.fit(stacked_predictions, y_test)

# Final Ensemble Prediction
final_preds = meta_model.predict(stacked_predictions)

In [5]:
def evaluate(y_true, y_pred, model_name):
    # Convert back from log scale to real currency
    true_val = np.expm1(y_true)
    pred_val = np.expm1(y_pred)
    
    rmse = np.sqrt(mean_squared_error(true_val, pred_val))
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} -> RMSE: ${rmse:,.2f} | R2 Score: {r2:.4f}")

print("--- Tabular Baseline Results ---")
evaluate(y_test, rf_preds, "Random Forest")
evaluate(y_test, xgb_preds, "XGBoost")
evaluate(y_test, cat_preds, "CatBoost")
evaluate(y_test, final_preds, "Stacked Ensemble (KNN)")

--- Tabular Baseline Results ---
Random Forest -> RMSE: $198,789.26 | R2 Score: 0.6757
XGBoost -> RMSE: $192,339.62 | R2 Score: 0.6809
CatBoost -> RMSE: $190,592.69 | R2 Score: 0.6937
Stacked Ensemble (KNN) -> RMSE: $172,165.22 | R2 Score: 0.7621


In [6]:
# Create interactions between the top 3 most important features
top_features = ['grade', 'log_sqft_living', 'bathrooms']

for i in range(len(top_features)):
    for j in range(i + 1, len(top_features)):
        col_name = f"{top_features[i]}_x_{top_features[j]}"
        df_final[col_name] = df_final[top_features[i]] * df_final[top_features[j]]

# Update X_train and X_test with these new features
X = df_final.drop(columns=['log_price', 'lat', 'long'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# =================================================================
# 1. DATA LOADING & PREPROCESSING FUNCTION
# =================================================================
def full_preprocessing(df, is_train=True):
    # Standardize column names
    df.columns = df.columns.str.strip()
    
    # 1. Feature Engineering: Temporal & Boolean
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['sell_year'] = df['date'].dt.year
    df['sell_month'] = df['date'].dt.month
    df['house_age'] = 2024 - df['yr_built']
    df['is_renovated'] = (df['yr_renovated'] > 0).astype(int)
    
    # 2. Log Transformations for Skewed Area Columns
    area_cols = ['sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']
    for col in area_cols:
        df[f'log_{col}'] = np.log1p(df[col])
        
    # 3. Target Transformation (Training Only)
    if is_train:
        df['log_price'] = np.log1p(df['price'])
        
    # 4. Feature Selection based on Correlation > 0.1 (Determined in EDA)
    # We explicitly keep 'lat' and 'long' for the satellite imagery stage
    retained_features = [
        'bedrooms', 'bathrooms', 'floors', 'waterfront', 'view', 'grade', 
        'lat', 'long', 'is_renovated', 'log_sqft_living', 'log_sqft_lot', 
        'log_sqft_above', 'log_sqft_basement', 'log_sqft_living15', 'log_sqft_lot15'
    ]
    
    if is_train:
        return df[retained_features], df['log_price']
    else:
        return df[retained_features]

# Load original files
raw_train = pd.read_csv("train(1)(train(1)).csv")
raw_test = pd.read_csv("test2(test(1)).csv") # Adjust filename if necessary

X, y_log = full_preprocessing(raw_train, is_train=True)
X_test = full_preprocessing(raw_test, is_train=False)

print(f"Dataset ready. Features: {X.shape[1]}")

# =================================================================
# 2. STAGE 1: LEAKAGE-SAFE BASE ENSEMBLE
# =================================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_base = np.zeros(len(X))
test_base = np.zeros(len(X_test))

print("\n🚀 Training Stage 1: Base Models (XGB, Cat, LGBM)...")

for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y_log), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y_log.iloc[tr_idx], y_log.iloc[va_idx]
    
    # Models tuned for log-price prediction
    m1 = XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=7, random_state=42)
    m2 = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=7, verbose=0, random_seed=42)
    m3 = LGBMRegressor(n_estimators=1000, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1)
    
    m1.fit(X_tr, y_tr)
    m2.fit(X_tr, y_tr)
    m3.fit(X_tr, y_tr)
    
    # Blend: 40% XGB, 40% Cat, 20% LGBM
    fold_preds = (0.4 * m1.predict(X_va)) + (0.4 * m2.predict(X_va)) + (0.2 * m3.predict(X_va))
    oof_base[va_idx] = fold_preds
    
    # Average test predictions across folds
    test_base += (0.4 * m1.predict(X_test) + 0.4 * m2.predict(X_test) + 0.2 * m3.predict(X_test)) / 5
    print(f"   Fold {fold} complete.")

print(f"Base Ensemble R2: {r2_score(y_log, oof_base):.4f}")

# =================================================================
# 3. STAGE 2: RESIDUAL LEARNING (ERROR CORRECTION)
# =================================================================
residuals = y_log - oof_base
oof_res = np.zeros(len(X))
test_res = np.zeros(len(X_test))

print("\n🧠 Training Stage 2: Residual Correction...")

for fold, (tr_idx, va_idx) in enumerate(kf.split(X, residuals), 1):
    X_tr_res, X_va_res = X.iloc[tr_idx], X.iloc[va_idx]
    res_tr = residuals.iloc[tr_idx]
    
    # Using LightGBM as a meta-learner for residuals
    res_model = LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=15, random_state=42, verbose=-1)
    res_model.fit(X_tr_res, res_tr)
    
    oof_res[va_idx] = res_model.predict(X_va_res)
    test_res += res_model.predict(X_test) / 5

# =================================================================
# 4. FINAL INTEGRATION & SAVING
# =================================================================
final_oof_preds = oof_base + oof_res
final_test_preds_log = test_base + test_res

print("\n" + "="*30)
print(f"FINAL TABULAR R2: {r2_score(y_log, final_oof_preds):.4f}")
print("="*30)

# Inverse log transformation to get USD prices
final_prices = np.expm1(final_test_preds_log)

# Format for Submission
submission = pd.DataFrame({
    'id': raw_test['id'],
    'predicted_price': final_prices
})

submission.to_csv("submission_tabular_final.csv", index=False)
print("💾 Prediction file saved as 'submission_tabular_final.csv'")

Dataset ready. Features: 15

🚀 Training Stage 1: Base Models (XGB, Cat, LGBM)...
   Fold 1 complete.
   Fold 2 complete.
   Fold 3 complete.
   Fold 4 complete.
   Fold 5 complete.
Base Ensemble R2: 0.8990

🧠 Training Stage 2: Residual Correction...

FINAL TABULAR R2: 0.8881
💾 Prediction file saved as 'submission_tabular_final.csv'


In [14]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# 1. Load and Refit Base Model
# Assuming X and y_log are already preprocessed from the original train(1).xlsx
print("🚀 Training Base Tabular Model...")

# Initialize Models with the parameters that gave 0.8990
m1_base = XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=7, random_state=42)
m2_base = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=7, verbose=0, random_seed=42)
m3_base = LGBMRegressor(n_estimators=1000, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1)

# Fit on full dataset for the "Save" version
m1_base.fit(X, y_log)
m2_base.fit(X, y_log)
m3_base.fit(X, y_log)

# 2. Save Base Model
base_path = "models/base_tabular"
os.makedirs(base_path, exist_ok=True)

m1_base.save_model(f"{base_path}/xgb_base.json")
m2_base.save_model(f"{base_path}/cat_base.cbm")
m3_base.booster_.save_model(f"{base_path}/lgbm_base.txt")
joblib.dump(X.columns.tolist(), f"{base_path}/features.pkl")

print(f"✅ Base Tabular Model Saved.")

🚀 Training Base Tabular Model...
✅ Base Tabular Model Saved.


In [20]:
import os
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# ==========================================
# 1. PREPROCESSING & DATA LOADING
# ==========================================
def preprocess_data(df, is_train=True):
    print(f"🛠️  Preprocessing {'Training' if is_train else 'Test'} Data...")
    df = df.copy()
    df.columns = df.columns.str.strip()
    
    # Feature Engineering
    df['is_renovated'] = (df['yr_renovated'] > 0).astype(int)
    area_cols = ['sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']
    for col in area_cols:
        df[f'log_{col}'] = np.log1p(df[col])
    
    if is_train:
        df['log_price'] = np.log1p(df['price'])
        
    features = [
        'bedrooms', 'bathrooms', 'floors', 'waterfront', 'view', 'grade', 
        'lat', 'long', 'is_renovated', 'log_sqft_living', 'log_sqft_lot', 
        'log_sqft_above', 'log_sqft_basement', 'log_sqft_living15', 'log_sqft_lot15'
    ]
    return (df[features], df['log_price']) if is_train else df[features]

# Load Data
train_raw = pd.read_csv("train(1)(train(1)).csv")
test_raw = pd.read_csv("test2(test(1)).csv")

X_tab, y_log = preprocess_data(train_raw, is_train=True)
X_test_tab = preprocess_data(test_raw, is_train=False)

# ==========================================
# 2. TRAINING FUNCTION (XGB + CAT + LGBM)
# ==========================================
def train_and_evaluate(X_in, y_in, model_label="Model"):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(X_in))
    
    print(f"\n🚀 Starting Training Pipeline for: {model_label}")
    # Using tqdm for progress bar across K-Folds
    for fold, (tr_idx, va_idx) in enumerate(tqdm(list(kf.split(X_in)), desc=f"Training {model_label} Folds"), 1):
        X_tr, X_va = X_in.iloc[tr_idx], X_in.iloc[va_idx]
        y_tr, y_va = y_in.iloc[tr_idx], y_in.iloc[va_idx]
        
        m1 = XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=7, random_state=42)
        m2 = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=7, verbose=0, random_seed=42)
        m3 = LGBMRegressor(n_estimators=1000, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1)
        
        m1.fit(X_tr, y_tr)
        m2.fit(X_tr, y_tr)
        m3.fit(X_tr, y_tr)
        
        fold_preds = (0.4 * m1.predict(X_va)) + (0.4 * m2.predict(X_va)) + (0.2 * m3.predict(X_va))
        oof[va_idx] = fold_preds
        
    score = r2_score(y_in, oof)
    print(f"📊 {model_label} Cross-Validation R2 Score: {score:.5f}")
    
    print(f"🔄 Refitting {model_label} on full dataset for final production...")
    m1.fit(X_in, y_in); m2.fit(X_in, y_in); m3.fit(X_in, y_in)
    return score, (m1, m2, m3)

# ==========================================
# 3. COMPETITION: TABULAR VS HYBRID
# ==========================================
# Stage A: Tabular
base_r2, base_models = train_and_evaluate(X_tab, y_log, "Tabular Baseline")

# Stage B: Hybrid
print("\n🧬 Preparing Hybrid Features (Tabular + Visual Embeddings)...")
pca = PCA(n_components=50, random_state=42)
vis_pca_train = pca.fit_transform(visual_embeddings) 
X_hybrid_train = pd.concat([X_tab.reset_index(drop=True), pd.DataFrame(vis_pca_train)], axis=1)

hybrid_r2, hybrid_models = train_and_evaluate(X_hybrid_train, y_log, "Hybrid Multimodal")

# ==========================================
# 4. FINAL PREDICTION LOGIC
# ==========================================
print("\n" + "="*50)
print(f"🏆 FINAL PERFORMANCE REPORT")
print(f"   - Tabular Baseline R2: {base_r2:.5f}")
print(f"   - Hybrid Multimodal R2: {hybrid_r2:.5f}")
print("="*50)

if hybrid_r2 > base_r2:
    print("📈 Decision: Hybrid model is superior. Processing satellite imagery for test set...")
    # Using your existing extract_visual_features function
    vis_embeddings_test = extract_visual_features(X_test_tab, "property_image_test111")
    
    print("🪄 Applying PCA transformation to test visual embeddings...")
    vis_pca_test = pca.transform(vis_embeddings_test)
    
    X_test_final = pd.concat([X_test_tab.reset_index(drop=True), pd.DataFrame(vis_pca_test)], axis=1)
    final_models = hybrid_models
    print("✅ Hybrid test dataset successfully prepared.")
else:
    print("📉 Decision: Tabular model performed better. Skipping visual data for predictions.")
    X_test_final = X_test_tab
    final_models = base_models

# ==========================================
# 5. SUBMISSION GENERATION
# ==========================================
print(f"\n🔮 Generating final predictions using the winning model...")
m1, m2, m3 = final_models
preds_log = (0.4 * m1.predict(X_test_final)) + (0.4 * m2.predict(X_test_final)) + (0.2 * m3.predict(X_test_final))
final_prices = np.expm1(preds_log)

submission = pd.DataFrame({'id': test_raw['id'], 'predicted_price': final_prices})
submission.to_csv("final_multimodal_predictions.csv", index=False)

print("\n" + "✨" * 20)
print("💾 DONE! Predictions saved: 'final_multimodal_predictions.csv'")
print("✨" * 20)

🛠️  Preprocessing Training Data...
🛠️  Preprocessing Test Data...

🚀 Starting Training Pipeline for: Tabular Baseline


Training Tabular Baseline Folds: 100%|██████████| 5/5 [00:41<00:00,  8.32s/it]


📊 Tabular Baseline Cross-Validation R2 Score: 0.89898
🔄 Refitting Tabular Baseline on full dataset for final production...

🧬 Preparing Hybrid Features (Tabular + Visual Embeddings)...

🚀 Starting Training Pipeline for: Hybrid Multimodal


Training Hybrid Multimodal Folds: 100%|██████████| 5/5 [03:08<00:00, 37.67s/it]


📊 Hybrid Multimodal Cross-Validation R2 Score: 0.89353
🔄 Refitting Hybrid Multimodal on full dataset for final production...

🏆 FINAL PERFORMANCE REPORT
   - Tabular Baseline R2: 0.89898
   - Hybrid Multimodal R2: 0.89353
📉 Decision: Tabular model performed better. Skipping visual data for predictions.

🔮 Generating final predictions using the winning model...

✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨
💾 DONE! Predictions saved: 'final_multimodal_predictions.csv'
✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨
